# Measuring the temperature of MoM-BH\*-1

**Goal:** estimate the temperature of MoM-BH\*-1's glowing gas surface by fitting a **blackbody**, the idealised glow of a hot object, to its spectrum. It's the same idea as estimating a star's temperature from the shape of its spectrum.

**What to compare with:**
- Curtis et al. (2026) fitted MoM-BH\*-1 with stellar-atmosphere models and got **about 6,500 K** (range roughly 6,200–6,650 K).
- Other black hole stars: PAN-BH\*-1 at about 4,800 K, and the lensed little red dot RX1 at about 4,000 K.
- Quasi-star models predict surfaces between roughly 4,000 and 9,000 K.

A blackbody is simpler than a real atmosphere, so don't expect to match 6,500 K exactly. A difference of several hundred degrees is normal.

**How to use:** run the grey code cells from top to bottom with **Shift + Enter**.

*Data: Naidu et al., Zenodo record 15059215 (CC BY 4.0).*

## Step 1 — Download the black hole star's spectrum

This is the same file as before, `spec_150135.fits`, which you confirmed is MoM-BH\*-1. Colab forgets files between sessions, so download it again.

In [ ]:
!wget -q -O spec_150135.fits "https://zenodo.org/records/15059215/files/mom-uds01-v3_prism-clear_5224_150135.spec.fits?download=1"

import os
size_kb = os.path.getsize("spec_150135.fits") / 1024 if os.path.exists("spec_150135.fits") else 0
print(f"spec_150135.fits: {size_kb:.0f} KB ->", "OK" if size_kb > 300 else "download failed, run this cell again in a few minutes")

## Step 2 — Load the spectrum and choose which parts to fit

A blackbody is a smooth curve, so we only fit the parts of the spectrum that should be smooth:

- **We start at 4,150 Å (rest frame).** The Balmer break is *not* part of a blackbody: it's light removed by the dense hydrogen. So we skip it and everything bluer.
- **We skip the spectral lines** (Hγ, Hβ with [O III], and He I), because lines are extra light or missing light on top of the smooth glow.

The cell converts observed wavelengths to **rest-frame** wavelengths (divide by 1 + z), so we look at the light as the object emitted it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.table import Table

Z = 7.7569                  # redshift of MoM-BH*-1
FIT_START = 4150            # rest-frame Å: start fitting just redward of the Balmer break
LINE_MASKS = [(4290, 4400, "Hγ"), (4770, 5040, "Hβ + [O III]"), (5830, 5930, "He I")]   # rest-frame Å


def load_spectrum(name):
    '''Return wavelength (µm), flux and error (µJy), and the sky position (RA, Dec).'''
    with fits.open(name) as f:
        ext = "SPEC1D" if "SPEC1D" in f else 1
        tab = Table(f[ext].data)
        ra = dec = None
        for hdu in f:
            if "SRCRA" in hdu.header:
                ra, dec = hdu.header["SRCRA"], hdu.header["SRCDEC"]
                break
    cols = {c.lower(): c for c in tab.colnames}
    err_col = cols.get("err", cols.get("full_err"))
    wave = np.array(tab[cols["wave"]], dtype=float)
    flux = np.array(tab[cols["flux"]], dtype=float)
    err = np.array(tab[err_col], dtype=float)
    if np.nanmedian(wave) > 100:                    # wavelengths in Å? convert to µm
        wave = wave / 1e4
    good = np.isfinite(wave) & np.isfinite(flux) & np.isfinite(err) & (err > 0)
    return wave[good], flux[good], err[good], ra, dec


wave, flux, err, ra, dec = load_spectrum("spec_150135.fits")
rest = wave * 1e4 / (1 + Z)       # rest-frame wavelength in Å
flux_nJy = flux * 1000            # µJy -> nJy
err_nJy = err * 1000

print(f"Spectrum covers {rest.min():.0f} - {rest.max():.0f} Å in the rest frame")

## Step 3 — Fit the blackbody

The method is simple and transparent:

1. Try **every temperature from 2,500 K to 15,000 K**, in steps of 10 K.
2. For each temperature, slide the blackbody curve up or down to find the height that best matches the data.
3. Measure the mismatch with **chi-square (χ²)**: the sum of (data − model)² / error² over all fitted pixels. Lower is better.
4. The temperature with the lowest χ² is the best fit. The **error range** is every temperature whose χ² is within 1 of the best. If the fit isn't perfect (reduced χ² above 1), the range is widened to match.

**Reduced χ²** is χ² divided by the number of data points minus 2 (for the two fitted numbers). A value of about 1 means the blackbody fits as well as the noise allows. A value much bigger than 1 means the real spectrum has shapes a pure blackbody can't make, or the error bars are too small.

The cell also checks the fit against **MIRI photometry at a longer wavelength that the fit never sees** (from the paper's table, observed in January 2023).

In [ ]:
HC_OVER_K = 1.4388e8   # Planck's constant x speed of light / Boltzmann's constant, in Å·K


def blackbody_shape(lam, T, lam_ref=5000.0):
    '''Blackbody brightness per unit frequency (B_nu) at wavelength lam (Å),
    scaled so it equals 1 at lam_ref. Only the SHAPE depends on temperature T.'''
    x = HC_OVER_K / (lam * T)
    x_ref = HC_OVER_K / (lam_ref * T)
    return (lam_ref / lam) ** 3 * np.expm1(x_ref) / np.expm1(x)


def fit_blackbody(start=FIT_START):
    '''Try every temperature from 2,500 to 15,000 K and return the best fit and its error range.'''
    use = rest >= start                                   # redward of the start wavelength...
    for lo, hi, name in LINE_MASKS:
        use &= ~((rest >= lo) & (rest <= hi))             # ...and outside the masked lines
    lam, f, e = rest[use], flux_nJy[use], err_nJy[use]
    w = 1 / e**2
    temps = np.arange(2500, 15001, 10)
    chi2 = np.empty(temps.size)
    heights = np.empty(temps.size)
    for i, T in enumerate(temps):
        m = blackbody_shape(lam, T)
        heights[i] = np.sum(w * f * m) / np.sum(w * m * m)   # best height for this temperature
        chi2[i] = np.sum(w * (f - heights[i] * m) ** 2)
    best = np.argmin(chi2)
    chi2_red = chi2[best] / (lam.size - 2)
    delta = max(1.0, chi2_red)                            # widen the range if the fit isn't perfect
    ok = temps[chi2 <= chi2[best] + delta]
    return dict(use=use, n=int(use.sum()), temps=temps, chi2=chi2, best=best, delta=delta,
                T=temps[best], A=heights[best], T_low=ok.min(), T_high=ok.max(), chi2_red=chi2_red)


fit = fit_blackbody()
T_best, A_best, T_low, T_high = fit["T"], fit["A"], fit["T_low"], fit["T_high"]

print(f"Pixels used in the fit: {fit['n']}")
print(f"Best-fit temperature:   {T_best} K   (range {T_low} - {T_high} K)")
print(f"Reduced chi-square:     {fit['chi2_red']:.2f}")
print(f"Blackbody brightness at 5000 Å: {A_best:.0f} nJy")
if T_low == fit["temps"].min() or T_high == fit["temps"].max():
    print("Note: the range touches the edge of the grid, so the temperature isn't well pinned down on that side.")

# Check against a data point the fit never saw: MIRI F770W photometry from the paper (observed Jan 2023)
MIRI_LAM = 7.64e4 / (1 + Z)       # F770W wavelength (7.64 µm) in rest-frame Å
MIRI_FLUX, MIRI_ERR = 465, 52     # nJy
predicted = A_best * blackbody_shape(MIRI_LAM, T_best)
print(f"\nMIRI check at {MIRI_LAM:.0f} Å: measured {MIRI_FLUX} ± {MIRI_ERR} nJy, blackbody predicts {predicted:.0f} nJy")

## Step 4 — Plot the fit

**Top plot:** the black line is the spectrum and the green dots are the pixels used in the fit. The red curve is the best-fit blackbody, and yellow strips are the masked lines.

- Look at the region left of the **Balmer limit**. The data fall far below the red curve there. That missing light is what the dense hydrogen absorbs, and it's the Balmer break you measured earlier.
- The **orange square** is the MIRI point the fit never saw. If it lands near the red curve, the blackbody works over a wide wavelength range. MIRI observed almost two years before the spectrum, and the object may have changed brightness since, so a small mismatch is expected.

**Bottom plot:** how χ² changes with temperature. The shaded band is the error range.

Both plots are saved as **MoM-BH1_temperature_fit.png** (folder icon on the left → ⋮ → Download).

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8.5), gridspec_kw={"height_ratios": [2.2, 1]})

# --- top: spectrum and blackbody ---
use = fit["use"]
ax1.step(rest, flux_nJy, where="mid", color="black", lw=1, label="MoM-BH*-1 (JWST prism)")
ax1.fill_between(rest, flux_nJy - err_nJy, flux_nJy + err_nJy, step="mid", color="gray", alpha=0.3)
ax1.plot(rest[use], flux_nJy[use], ".", color="tab:green", ms=5, label="pixels used in the fit")
lam_model = np.linspace(3000, 9500, 1000)
ax1.plot(lam_model, A_best * blackbody_shape(lam_model, T_best), color="tab:red", lw=2,
         label=f"blackbody, T = {T_best} K")
for k, (lo, hi, name) in enumerate(LINE_MASKS):
    ax1.axvspan(lo, hi, color="gold", alpha=0.35, label="masked lines" if k == 0 else None)
ax1.axvline(3647, color="gray", ls="--", lw=0.8)
ax1.errorbar(MIRI_LAM, MIRI_FLUX, yerr=MIRI_ERR, fmt="s", color="tab:orange", ms=7,
             label="MIRI F770W (Jan 2023, not fitted)")
show = (rest > 3000) & (rest < 9500)
top = 1.15 * max(np.max(flux_nJy[show]), MIRI_FLUX + MIRI_ERR)
ax1.set_ylim(-0.08 * top, top)
ax1.text(3647, 0.95 * top, "Balmer limit ", ha="right", va="top", fontsize=9, color="gray")
ax1.set_xlim(3000, 9500)
ax1.set_xlabel("Rest-frame wavelength (Å)")
ax1.set_ylabel("Brightness (nJy)")
ax1.set_title(f"MoM-BH*-1 blackbody fit: T = {T_best} K (range {T_low}-{T_high} K)")
ax1.legend(loc="lower right", fontsize=9)

# --- bottom: chi-square versus temperature ---
dchi2 = fit["chi2"] - fit["chi2"][fit["best"]]
ax2.plot(fit["temps"], dchi2, color="black")
ax2.axvspan(T_low, T_high, color="tab:red", alpha=0.2, label="error range")
ax2.axvline(T_best, color="tab:red", lw=1)
near = fit["temps"][dchi2 <= 30 * fit["delta"]]          # zoom in around the minimum
ax2.set_xlim(near.min() - 300, near.max() + 300)
ax2.set_ylim(0, 25 * fit["delta"])
ax2.set_xlabel("Temperature (K)")
ax2.set_ylabel("Δχ² (lower = better)")
ax2.legend(loc="upper right", fontsize=9)

plt.tight_layout()
fig.savefig("MoM-BH1_temperature_fit.png", dpi=150)
plt.show()

## Step 5 — How much does the answer depend on your choices?

A real measurement always involves choices. Here the main one is **where the fit starts**. This cell repeats the fit starting at four different wavelengths:

- If the temperature barely changes, your answer is robust.
- If it moves by hundreds of kelvin, that spread is a more honest error bar than the range from Step 3.

In [ ]:
for start in [4150, 4300, 4500, 5100]:
    r = fit_blackbody(start)
    print(f"fit starting at {start} Å: T = {r['T']} K  (range {r['T_low']}-{r['T_high']} K, "
          f"{r['n']} pixels, reduced chi-square {r['chi2_red']:.2f})")

## What your result means

- **Compare with about 6,500 K (Curtis et al.).** Their models include how hydrogen and other gas absorb light in the atmosphere, while a blackbody doesn't. So your number measures the same thing more simply, and a difference of several hundred degrees is expected.
- **Compare with other black hole stars** (about 4,000–4,800 K) and with quasi-star predictions (about 4,000–9,000 K). Is MoM-BH\*-1 hotter or cooler than the others?
- **Look at the reduced χ².** If it's well above 1, the spectrum isn't a perfect blackbody. Knowing where the data and the red curve disagree is useful information in itself.
- **Treat the Step 3 range as a minimum error.** Neighbouring pixels in a prism spectrum aren't fully independent, and a blackbody is only an approximation. The spread from Step 5 is usually the more honest uncertainty.

**Next step:** with the temperature and the object's total brightness, the Stefan–Boltzmann law L = 4πR²σT⁴ gives its **size**, which is one of the open questions.